<a href="https://colab.research.google.com/github/Gnoltd/BitcoinPredictionResearch/blob/main/%5BCRYPTO%5D_TECHNICAL_INDICATORS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# CRYPTO FORECASTING: END-TO-END DATA PIPELINE, FEATURE SELECTION, AND LSTM
# ==============================================================================

import os
import random
import hashlib
import requests
import re
from dataclasses import dataclass, field
from datetime import datetime
from typing import List

import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from boruta import BorutaPy
from google.colab import drive


# ==============================================================================
# CONFIGURATION
# ==============================================================================

@dataclass
class Config:
    coin: str = 'bitcoin'
    start_date: str = '2013-01-01'
    end_date: str = '2025-12-31'
    train_start: str = '2016-01-01'
    train_end: str = '2023-12-31'
    timeframes: List[int] = field(default_factory=lambda: [1, 7, 14])

    sequence_length: int = 14
    epochs: int = 100
    batch_size: int = 32

    features_to_fetch: List[str] = field(default_factory=lambda: [
        'transactions', 'size', 'sentbyaddress', 'transactionfees', 'blocktime',
        'difficulty', 'hashrate', 'transactionvalue', 'mediantransactionvalue',
        'profitability', 'activeaddresses', 'sentinusd', 'top100cap',
        'fee-to-reward-ratio', 'mediantransactionfee', 'price'
    ])

    base_dir: str = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'

    @property
    def raw_dir(self):      return os.path.join(self.base_dir, 'RawData')
    @property
    def proc_dir(self):     return os.path.join(self.base_dir, 'ProcessedData')
    @property
    def feat_sel_dir(self): return os.path.join(self.base_dir, 'Features_Selection_Golden')
    @property
    def lstm_out_dir(self): return os.path.join(self.base_dir, 'LSTM_Results')


# ==============================================================================
# HELPERS
# ==============================================================================

def apply_modwt(df: pd.DataFrame) -> pd.DataFrame:
    df_denoised = df.copy()
    data_length = len(df)
    padded_length = int(np.ceil(data_length / 32.0)) * 32
    for col in df.columns:
        signal = df[col].values
        padded_signal = np.pad(signal, (0, padded_length - data_length), mode='edge')
        coeffs = pywt.swt(padded_signal, 'db2', level=5)
        denoised_coeffs = [(approx, np.zeros_like(detail)) for approx, detail in coeffs]
        denoised_padded = pywt.iswt(denoised_coeffs, 'db2')
        df_denoised[col] = denoised_padded[:data_length]
    return df_denoised


def compute_rsi(series: pd.Series, window: int) -> pd.Series:
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return (100 - (100 / (1 + rs))).fillna(50)


def calculate_vif_recursive(X: pd.DataFrame, threshold: float = 10.0) -> List[str]:
    Xc = X.replace([np.inf, -np.inf], np.nan).dropna()
    features = Xc.columns.tolist()
    while len(features) >= 2:
        vif_vals = [variance_inflation_factor(Xc[features].values, i) for i in range(len(features))]
        max_vif = max(vif_vals)
        if max_vif > threshold:
            features.pop(vif_vals.index(max_vif))
        else:
            break
    return features


def create_sequences(data: pd.DataFrame, target: pd.Series, seq_length: int):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        xs.append(data.iloc[i:(i + seq_length)].values)
        ys.append(target.iloc[i + seq_length])
    return np.array(xs), np.array(ys)





# ==============================================================================
# SECTION 1: DATA CRAWLING
# ==============================================================================

def fetch_bitinfocharts_data(feature: str, coin: str):
    url = f"https://bitinfocharts.com/comparison/{coin}-{feature}.html#alltime"
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
    response.raise_for_status()

    content_hash = hashlib.sha256(response.content).hexdigest()
    matches = re.findall(r'\[new Date\("(.*?)"\),(.*?)\]', response.text)

    dates, values = [], []
    for match in matches:
        dates.append(pd.to_datetime(match[0]))
        val = match[1]
        values.append(float(val) if val != 'null' else np.nan)

    df = pd.DataFrame({'Date': dates, feature: values}).set_index('Date')
    return df, content_hash


def crawl_raw_data(cfg: Config) -> pd.DataFrame:
    print("--- STEP 1: CRAWLING RAW DATA ---")
    raw_csv_path = os.path.join(cfg.raw_dir, f'{cfg.coin}_raw_data.csv')

    if os.path.exists(raw_csv_path):
        print("Raw data already exists. Loading from disk.")
        return pd.read_csv(raw_csv_path, index_col='Date', parse_dates=True)

    df_raw = pd.DataFrame()
    for feature in cfg.features_to_fetch:
        df_temp, _ = fetch_bitinfocharts_data(feature, cfg.coin)
        df_raw = df_temp if df_raw.empty else df_raw.join(df_temp, how='outer')
        print(f"Fetched: {feature}")

    if 'price' in df_raw.columns:
        df_raw.rename(columns={'price': 'Close'}, inplace=True)
    df_raw.to_csv(raw_csv_path)
    return df_raw


# ==============================================================================
# SECTION 2: PRE-PROCESSING & FEATURE ENGINEERING
# ==============================================================================

def preprocess_and_engineer(df_raw: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    print("\n--- STEP 2: PRE-PROCESSING & ENGINEERING ---")
    engineered_path = os.path.join(cfg.proc_dir, f'{cfg.coin}_engineered.csv')

    if os.path.exists(engineered_path):
        print("Engineered data already exists. Loading from disk.")
        return pd.read_csv(engineered_path, index_col='Date', parse_dates=True)

    df_raw = df_raw.reindex(pd.date_range(start=cfg.start_date, end=cfg.end_date, freq='D'))
    df_raw.index.name = 'Date'
    df_full = df_raw.copy()

    y_raw_close = df_full['Close'].copy()
    df_full.interpolate(method='cubicspline', inplace=True)
    df_full.ffill(inplace=True)
    df_full.bfill(inplace=True)
    df_full = df_full.dropna(subset=['Close'])
    y_raw_close = y_raw_close.reindex(df_full.index)

    train_mask = (df_full.index >= cfg.train_start) & (df_full.index <= cfg.train_end)
    df_train_only = df_full.loc[train_mask]

    feature_cols = [col for col in df_full.columns if col != 'Close']
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(df_train_only[feature_cols])
    df_features_scaled = pd.DataFrame(
        scaler.transform(df_full[feature_cols]),
        index=df_full.index,
        columns=feature_cols
    )

    df_preprocessed = apply_modwt(df_features_scaled)
    df_preprocessed['y_raw_close_price'] = y_raw_close

    windows = [7, 14]
    features_dfs = [df_preprocessed[feature_cols].copy()]

    for col in feature_cols:
        series = df_preprocessed[col]
        df_temp = pd.DataFrame(index=df_preprocessed.index)
        for w in windows:
            df_temp[f'{col}_{w}_SMA'] = series.rolling(window=w).mean()
            df_temp[f'{col}_{w}_EMA'] = series.ewm(span=w, adjust=False).mean()
            weights = np.arange(1, w + 1)
            df_temp[f'{col}_{w}_WMA'] = series.rolling(window=w).apply(
                lambda x: np.dot(x, weights) / weights.sum(), raw=True
            )
            df_temp[f'{col}_{w}_STD'] = series.rolling(window=w).std()
            df_temp[f'{col}_{w}_ROC'] = series.pct_change(periods=w) * 100
            df_temp[f'{col}_{w}_RSI'] = compute_rsi(series, w)
        features_dfs.append(df_temp)

    df_engineered = pd.concat(features_dfs, axis=1)
    df_engineered.replace([np.inf, -np.inf], np.nan, inplace=True)

    for tf in cfg.timeframes:
        # Log return over the forecast horizon: log(price_{t+tf} / price_t)
        df_engineered[f'Target_{tf}d'] = np.log(y_raw_close.shift(-tf) / y_raw_close)

    df_engineered.dropna(inplace=True)
    df_engineered.to_csv(engineered_path)
    print(f"Engineered data saved: {df_engineered.shape[1]} features, {len(df_engineered)} rows.")
    return df_engineered


# ==============================================================================
# SECTION 3: FEATURE SELECTION (THE GOLDEN PIPELINE)
# ==============================================================================

import textwrap as _textwrap


def _fs_log(stage, label, features, log):
    """Record and print a feature selection stage count."""
    n       = len(features)
    dropped = log[-1]['n_features'] - n if log else 0
    print(f'  Stage {stage} | {label:<45} | {n:>4} features  (−{dropped})')
    log.append({'stage': stage, 'label': label, 'n_features': n, 'dropped': dropped})


def _fs_plot_funnel(log, tf, out_dir):
    """Horizontal funnel chart for a single TF."""
    import seaborn as _sns
    palette = _sns.color_palette('muted')
    labels  = [_textwrap.fill(r['label'], 28) for r in log]
    values  = [r['n_features'] for r in log]
    colors  = [palette[0] if i == 0
               else ('#e74c3c' if i == len(log) - 1 else palette[1])
               for i in range(len(log))]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(labels[::-1], values[::-1],
                   color=colors[::-1], edgecolor='white', height=0.55)
    for bar, val in zip(bars, values[::-1]):
        ax.text(val + 1, bar.get_y() + bar.get_height() / 2,
                str(val), va='center', fontsize=10, fontweight='bold')
    ax.set_xlabel('Number of Features')
    ax.set_title(f'Bitcoin — Feature Reduction Funnel  (TF{tf}d)',
                 fontsize=12, fontweight='bold')
    ax.set_xlim(0, max(values) * 1.15)
    fig.tight_layout()
    path = os.path.join(out_dir, f'FeatureReduction_Funnel_TF{tf}.png')
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'  Funnel chart saved  ->  {path}')


def _fs_plot_combined(all_logs, out_dir):
    """Grouped bar chart comparing all TFs at each stage."""
    import seaborn as _sns
    palette      = _sns.color_palette('muted')
    first_log    = next(iter(all_logs.values()))
    stage_labels = [_textwrap.fill(r['label'], 22) for r in first_log]
    tfs          = list(all_logs.keys())
    n_tfs        = len(tfs)
    x            = np.arange(len(stage_labels))
    width        = 0.22
    offset       = np.linspace(-(n_tfs - 1) * width / 2,
                                (n_tfs - 1) * width / 2, n_tfs)

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, tf in enumerate(tfs):
        values = [r['n_features'] for r in all_logs[tf]]
        bars   = ax.bar(x + offset[i], values, width,
                        label=f'TF{tf}d', color=palette[i], edgecolor='white')
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 1, str(val),
                    ha='center', va='bottom', fontsize=7, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(stage_labels, rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('Number of Features')
    ax.set_title('Bitcoin — Feature Reduction at Each Stage  (TF1 vs TF7 vs TF14)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    fig.tight_layout()
    path = os.path.join(out_dir, 'FeatureReduction_Funnel_ALL.png')
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'  Combined chart saved  ->  {path}')


def _run_fs_single_tf(df_eng, tf, cfg):
    """Run full feature selection for one timeframe. Returns step log."""
    print('\n' + '=' * 60)
    print(f'  FEATURE SELECTION — TF{tf}d')
    print('=' * 60)

    final_csv_path = os.path.join(cfg.feat_sel_dir, f'{cfg.coin}_GoldenFeatures_TF{tf}.csv')
    if os.path.exists(final_csv_path):
        print(f'  Golden features already exist for TF{tf}d — skipping.')
        print(f'  (Delete the file to re-run this timeframe.)')
        return []

    target_col      = f'Target_{tf}d'
    all_target_cols = [c for c in df_eng.columns if c.startswith('Target_')]
    targets_to_drop = [t for t in all_target_cols if t != target_col]
    valid_cols      = [c for c in df_eng.columns
                       if c not in targets_to_drop and c != target_col]

    df_tf = (df_eng[valid_cols + [target_col]]
             .replace([np.inf, -np.inf], np.nan)
             .dropna())
    if df_tf.empty:
        print('  Empty dataframe — skipping.')
        return []

    df_train = df_tf.loc[:cfg.train_end]
    X_train  = df_train.drop(columns=[target_col])
    y_train  = df_train[target_col]
    if X_train.empty:
        return []

    log = []

    # Stage 0: initial
    _fs_log(0, 'Initial candidate features', X_train.columns.tolist(), log)

    # Stage 1: VarianceThreshold
    print('\n  Stage 1: VarianceThreshold ...')
    selector_var = VarianceThreshold(threshold=0.0).fit(X_train)
    X_train      = X_train.loc[:, selector_var.get_support()]
    _fs_log(1, 'After VarianceThreshold', X_train.columns.tolist(), log)

    # Stage 2: RF permutation importance  (mean - 2*std > 0)
    print('\n  Stage 2: RF permutation importance (mean - 2*std > 0) ...')
    rf_quick    = RandomForestRegressor(n_estimators=100, max_depth=5,
                                        random_state=42, n_jobs=-1)
    rf_quick.fit(X_train, y_train)
    perm_result = permutation_importance(
        rf_quick, X_train, y_train,
        n_repeats=10, random_state=42, n_jobs=-1,
    )
    survivors_mask    = (perm_result.importances_mean - 2 * perm_result.importances_std) > 0
    step1_importances = dict(zip(
        X_train.columns[survivors_mask],
        perm_result.importances_mean[survivors_mask],
    ))
    X_train = X_train.loc[:, survivors_mask]
    if X_train.empty:
        return []
    _fs_log(2, 'After RF permutation (mean - 2*std > 0)', X_train.columns.tolist(), log)

    # Stage 3: Boruta
    print('\n  Stage 3: Boruta (this may take several minutes) ...')
    rf_boruta = RandomForestRegressor(n_jobs=-1, max_depth=5, random_state=42)
    boruta    = BorutaPy(rf_boruta, n_estimators='auto', verbose=0,
                         random_state=42, max_iter=50)
    boruta.fit(X_train.values, y_train.values)
    boruta_features = X_train.columns[boruta.support_ | boruta.support_weak_].tolist()
    if not boruta_features:
        print('  Boruta returned no features — skipping.')
        return []
    _fs_log(3, 'After Boruta (confirmed + tentative)', boruta_features, log)

    # Stage 4: Pearson collinearity pruning (r > 0.85)
    print('\n  Stage 4: Pearson collinearity pruning (r > 0.85) ...')
    X_boruta    = X_train[boruta_features]
    corr_matrix = X_boruta.corr().abs()
    upper_tri   = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop_pearson = set()
    for col in upper_tri.columns:
        for var in upper_tri.index[upper_tri[col] > 0.85]:
            if step1_importances.get(col, 0) < step1_importances.get(var, 0):
                to_drop_pearson.add(col)
            else:
                to_drop_pearson.add(var)
    features_after_pearson = [f for f in boruta_features if f not in to_drop_pearson]
    _fs_log(4, 'After Pearson pruning (r > 0.85)', features_after_pearson, log)

    # Stage 5: VIF recursive pruning (threshold = 10)
    print('\n  Stage 5: VIF recursive pruning (threshold = 10) ...')
    final_features = calculate_vif_recursive(
        X_train[features_after_pearson], threshold=10.0)
    _fs_log(5, 'After VIF pruning  -> FINAL', final_features, log)

    # Save golden features CSV
    df_tf[final_features + [target_col]].to_csv(final_csv_path)
    print(f'\n  Golden features saved  ->  {final_csv_path}')

    # Save step log CSV
    log_df       = pd.DataFrame(log)
    log_csv_path = os.path.join(cfg.feat_sel_dir,
                                f'{cfg.coin}_FeatureReduction_TF{tf}.csv')
    log_df.to_csv(log_csv_path, index=False)
    print(f'  Step log saved         ->  {log_csv_path}')

    # Print summary table
    print(f'\n{"─"*60}')
    print(f'  {"Stage":<6} {"Description":<45} {"N":>5} {"Dropped":>8}')
    print(f'{"─"*60}')
    for row in log:
        print(f'  {row["stage"]:<6} {row["label"]:<45} '
              f'{row["n_features"]:>5} {row["dropped"]:>8}')
    print(f'{"─"*60}')

    _fs_plot_funnel(log, tf, cfg.feat_sel_dir)
    print(f'\n  Done TF{tf}d — final feature count: {len(final_features)}')
    return log


def run_feature_selection(df_engineered: pd.DataFrame, cfg: Config) -> None:
    print("\n--- STEP 3: FEATURE SELECTION ---")

    all_logs = {}
    for tf in cfg.timeframes:
        log = _run_fs_single_tf(df_engineered, tf, cfg)
        if log:
            all_logs[tf] = log

    if len(all_logs) > 1:
        _fs_plot_combined(all_logs, cfg.feat_sel_dir)

    print('\n' + '=' * 60)
    print('  FEATURE SELECTION COMPLETE')
    print('=' * 60)

# ==============================================================================
# SECTION 4: LSTM MODELING & EVALUATION
# ==============================================================================

# ── Reproducibility — fix all random seeds so results are identical on re-runs
_SEED = 42


def _build_lstm_model(n_timesteps: int, n_features: int) -> Sequential:
    """
    Construct and compile the stacked LSTM architecture.

    Architecture
    ────────────
    LSTM(64) → Dropout(0.2)   ← captures long-range dependencies
    LSTM(32) → Dropout(0.2)   ← refines temporal patterns
    Dense(16, relu)            ← non-linear projection
    Dense(1)                   ← scalar log-return output (unbounded)
    """
    model = Sequential([
        LSTM(64, activation='tanh', return_sequences=True,
             input_shape=(n_timesteps, n_features)),
        Dropout(0.2),
        LSTM(32, activation='tanh', return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


def _evaluate_metrics(y_true_lr: np.ndarray, y_pred_lr: np.ndarray,
                      actual_prices: np.ndarray, predicted_prices: np.ndarray,
                      timeframe: int) -> pd.DataFrame:
    """
    Compute regression metrics in two spaces:
    - Log-return space : MAE, RMSE, R²  (the space the model was trained on)
    - Price space      : MAPE only      (log-returns near zero make MAPE
                                         undefined in log-return space;
                                         USD prices are always > 0)
    """
    mae  = mean_absolute_error(y_true_lr, y_pred_lr)
    rmse = np.sqrt(mean_squared_error(y_true_lr, y_pred_lr))
    r2   = r2_score(y_true_lr, y_pred_lr)
    mape = mean_absolute_percentage_error(actual_prices, predicted_prices) * 100   # %

    print(f"\n{'─'*47}")
    print(f"  REGRESSION METRICS  |  horizon = {timeframe}d")
    print(f"{'─'*47}")
    print(f"  [Log-Return Space]")
    print(f"  MAE   : {mae:>12.6f}")
    print(f"  RMSE  : {rmse:>12.6f}")
    print(f"  R²    : {r2:>12.4f}")
    print(f"  [Price Space]")
    print(f"  MAPE  : {mape:>11.2f}%")
    print(f"{'─'*47}\n")

    return pd.DataFrame({
        'Timeframe': [f'{timeframe}d'],
        'MAE':       [mae],
        'RMSE':      [rmse],
        'R2':        [r2],
        'MAPE_pct':  [mape],
    })


def _save_loss_chart(history, coin: str, tf_days: int, save_path: str) -> None:
    """Plot and save training / validation MSE loss curve."""
    plt.figure(figsize=(10, 6), dpi=300)
    plt.plot(history.history['loss'],     label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f"{coin.capitalize()} LSTM ({tf_days}d) — Training & Validation Loss")
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"  Loss chart saved  →  {save_path}")


def _save_forecast_chart(test_dates, actual_prices: np.ndarray,
                         predicted_prices: np.ndarray, metrics_row: pd.Series,
                         coin: str, tf_days: int, save_path: str) -> None:
    """Plot and save actual vs. predicted close-price chart (USD)."""
    plt.figure(figsize=(14, 7), dpi=300)
    plt.plot(test_dates, actual_prices,
             label='Actual Close Price', color='black', linewidth=1.5)
    plt.plot(test_dates, predicted_prices,
             label='LSTM Forecast', color='red', linestyle='--', linewidth=2)
    mape_str = (f"MAPE(price)={metrics_row['MAPE_pct']:.2f}%  |  "
                if 'MAPE_pct' in metrics_row.index else "")
    subtitle = (
        f"RMSE={metrics_row['RMSE']:.6f}  |  "
        f"MAE={metrics_row['MAE']:.6f}  |  "
        + mape_str +
        f"R²={metrics_row['R2']:.4f}"
    )
    plt.title(
        f"{coin.capitalize()} LSTM ({tf_days}d) Forecast — Golden Features\n" + subtitle
    )
    plt.xlabel("Date")
    plt.ylabel("Close Price (USD)")
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"  Forecast chart saved  →  {save_path}")


def _save_logreturn_chart(test_dates, y_true: np.ndarray, y_pred: np.ndarray,
                          metrics_row: pd.Series, coin: str, tf_days: int,
                          save_path: str) -> None:
    """Plot actual vs. predicted log-returns — the honest view of model accuracy."""
    plt.figure(figsize=(14, 7), dpi=300)
    plt.plot(test_dates, y_true,
             label='Actual Log-Return', color='black', linewidth=1.5)
    plt.plot(test_dates, y_pred,
             label='LSTM Predicted Log-Return', color='red',
             linestyle='--', linewidth=2)
    plt.axhline(0, color='gray', linewidth=0.8, linestyle=':')
    subtitle = (
        f"RMSE={metrics_row['RMSE']:.6f}  |  "
        f"MAE={metrics_row['MAE']:.6f}  |  "
        f"R²={metrics_row['R2']:.4f}"
    )
    plt.title(
        f"{coin.capitalize()} LSTM ({tf_days}d) — Log-Return Space\n" + subtitle
    )
    plt.xlabel("Date")
    plt.ylabel("Log-Return")
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"  Log-return chart saved  →  {save_path}")


def train_and_evaluate_lstm(cfg: Config) -> None:
    print("\n--- STEP 4: LSTM MODELING ---")

    # ── Fix #9: Seeds inside function to avoid module-level side effects ──────
    os.environ['PYTHONHASHSEED'] = str(_SEED)
    random.seed(_SEED)
    np.random.seed(_SEED)
    tf.random.set_seed(_SEED)
    tf.config.experimental.enable_op_determinism()

    metrics_summary_path = os.path.join(
        cfg.lstm_out_dir, f'{cfg.coin}_LSTM_Evaluation_Summary.csv')

    # Restore existing metrics to append, not overwrite
    all_metrics = (pd.read_csv(metrics_summary_path)
                   if os.path.exists(metrics_summary_path)
                   else pd.DataFrame())

    # Load close-price series from the engineered checkpoint.
    # Used to: (a) compute log-return targets and (b) reconstruct USD prices.
    engineered_path = os.path.join(cfg.proc_dir, f'{cfg.coin}_engineered.csv')
    close_price_all = pd.read_csv(
        engineered_path, index_col='Date', parse_dates=True)['y_raw_close_price']

    for tf_days in cfg.timeframes:
        print(f"\n--- TIMEFRAME: {tf_days} day(s) ---")

        # ── Per-timeframe output paths ────────────────────────
        predictions_path = os.path.join(
            cfg.lstm_out_dir, f'{cfg.coin}_LSTM_Predictions_TF{tf_days}.csv')
        chart_path       = os.path.join(
            cfg.lstm_out_dir, f'{cfg.coin}_LSTM_forecast_chart_TF{tf_days}.png')
        lr_chart_path    = os.path.join(
            cfg.lstm_out_dir, f'{cfg.coin}_LSTM_logreturn_chart_TF{tf_days}.png')
        loss_chart_path  = os.path.join(
            cfg.lstm_out_dir, f'{cfg.coin}_LSTM_loss_chart_TF{tf_days}.png')

        # Shared across resume and train branches
        actual_prices    = None
        predicted_prices = None
        test_dates       = None
        current_metrics  = None
        y_test_lr        = None
        y_pred_lr        = None

        # ── 4a. RESUME: load previously saved predictions ─────
        if os.path.exists(predictions_path):
            print(f"  Found saved predictions — loading from disk …")
            results_df = pd.read_csv(predictions_path,
                                     index_col='Date', parse_dates=True)

            required_cols = {'Actual_Log_Return', 'Predicted_Log_Return'}
            if not required_cols.issubset(results_df.columns):
                print("  WARNING: Prediction file missing required columns. "
                      "Deleting and re-training …")
                os.remove(predictions_path)
                for p in [chart_path, lr_chart_path]:
                    if os.path.exists(p):
                        os.remove(p)
            else:
                y_test_lr  = results_df['Actual_Log_Return'].values
                y_pred_lr  = results_df['Predicted_Log_Return'].values
                test_dates = results_df.index

                current_close = close_price_all.reindex(test_dates).dropna()
                if current_close.empty or len(current_close) != len(y_test_lr):
                    print("  Alignment mismatch — deleting cached files and re-training.")
                    os.remove(predictions_path)
                    for p in [chart_path, lr_chart_path]:
                        if os.path.exists(p):
                            os.remove(p)
                else:
                    actual_prices    = current_close.values * np.exp(y_test_lr)
                    predicted_prices = current_close.values * np.exp(y_pred_lr)

                    if (not all_metrics.empty and
                            f'{tf_days}d' in all_metrics['Timeframe'].values and
                            'MAPE_pct' in all_metrics.columns):
                        current_metrics = (
                            all_metrics[all_metrics['Timeframe'] == f'{tf_days}d']
                            .iloc[0]
                        )
                        print("  Metrics already on record — skipping re-evaluation.")
                    else:
                        # Fix #10: also persist metrics when computed on resume
                        metrics_df      = _evaluate_metrics(y_test_lr, y_pred_lr,
                                                            actual_prices, predicted_prices,
                                                            tf_days)
                        current_metrics = metrics_df.iloc[0]
                        all_metrics = pd.concat([all_metrics, metrics_df], ignore_index=True)
                        all_metrics.drop_duplicates(subset=['Timeframe'], keep='last', inplace=True)
                        all_metrics.to_csv(metrics_summary_path, index=False)
                        print(f"  Metrics summary updated  →  {metrics_summary_path}")

        # ── 4b. TRAIN: predictions file still doesn't exist ───
        if not os.path.exists(predictions_path):

            feature_file = os.path.join(
                cfg.feat_sel_dir, f'{cfg.coin}_GoldenFeatures_TF{tf_days}.csv')
            if not os.path.exists(feature_file):
                print(f"  Feature file not found: {feature_file}  →  skipping.")
                continue

            print(f"  Loading golden features from: {feature_file}")
            df_tf      = pd.read_csv(feature_file, index_col='Date', parse_dates=True)
            target_col = f'Target_{tf_days}d'

            drop_cols = [target_col] + [c for c in ['y_raw_close_price'] if c in df_tf.columns]
            X_raw = df_tf.drop(columns=drop_cols)

            # Fix #7: shift on full calendar-aligned close price, then reindex to df_tf.
            # This ensures shift(-tf_days) moves by actual calendar days, not row positions.
            # Without this, gaps in df_tf.index cause shift to skip wrong number of days.
            y_close_full = close_price_all  # full daily calendar from engineered file

            # Fix #7 (continued): build future_close from the full series, then align
            combined = pd.DataFrame({
                'close':        y_close_full.reindex(df_tf.index),
                'future_close': y_close_full.shift(-tf_days).reindex(df_tf.index),
            }, index=df_tf.index).dropna()

            if combined.empty:
                print(f"  No valid rows after alignment — skipping {tf_days}d.")
                continue

            X_raw        = X_raw.reindex(combined.index)
            y_log_return = np.log(combined['future_close'] / combined['close'])

            # ── Scale log-return target to [0, 1] ─────────────
            # Fit ONLY on training rows to prevent data leakage.
            target_scaler = MinMaxScaler(feature_range=(0, 1))

            # Fix #1: train_mask must be recomputed on y_log_return.index
            # (after combined.dropna()), not on the original df_tf.index.
            # If rows were dropped, train_mask.sum() would be wrong otherwise.
            train_mask = y_log_return.index <= cfg.train_end
            target_scaler.fit(y_log_return[train_mask].values.reshape(-1, 1))

            y_scaled = pd.Series(
                target_scaler.transform(
                    y_log_return.values.reshape(-1, 1)
                ).flatten(),
                index=y_log_return.index,
            )

            # ── Build sliding-window sequences ────────────────
            X_seq, y_seq = create_sequences(X_raw, y_scaled, cfg.sequence_length)

            # Fix #1: n_train derived from train_mask on aligned y_log_return index
            n_train = int(train_mask.sum()) - cfg.sequence_length
            if n_train <= 0:
                print(f"  Not enough training data for {tf_days}d — skipping.")
                continue

            X_train, y_train = X_seq[:n_train], y_seq[:n_train]
            X_test,  y_test  = X_seq[n_train:], y_seq[n_train:]

            print(f"  Train samples : {len(X_train):,}")
            print(f"  Test  samples : {len(X_test):,}")

            # ── Build and train model ─────────────────────────
            model = _build_lstm_model(
                n_timesteps=X_train.shape[1],
                n_features=X_train.shape[2],
            )

            early_stop = EarlyStopping(
                monitor='val_loss',
                patience=20,
                restore_best_weights=True,
            )

            print(f"  Training LSTM …  (max {cfg.epochs} epochs, patience=20)")
            history = model.fit(
                X_train, y_train,
                epochs=cfg.epochs,
                batch_size=cfg.batch_size,
                validation_split=0.1,
                callbacks=[early_stop],
                verbose=0,
            )
            print(f"  Stopped at epoch {len(history.history['loss'])} / {cfg.epochs}")

            _save_loss_chart(history, cfg.coin, tf_days, loss_chart_path)

            # ── Invert target scaling → log-return space ──────
            y_pred_lr = target_scaler.inverse_transform(
                model.predict(X_test, verbose=0)
            ).flatten()
            y_test_lr = target_scaler.inverse_transform(
                y_test.reshape(-1, 1)
            ).flatten()

            # Fix #1: test_dates derived from y_log_return.index (post-dropna)
            # so it stays in sync with the actual rows used for sequences.
            test_dates = y_log_return.index[n_train + cfg.sequence_length:]

            # ── Reconstruct USD prices ────────────────────────
            # Done before metrics so MAPE can be computed on price space.
            current_close = close_price_all.reindex(test_dates).dropna()
            if current_close.empty or len(current_close) != len(y_test_lr):
                print(f"  Alignment issue after training — skipping save for {tf_days}d.")
                continue

            actual_prices    = current_close.values * np.exp(y_test_lr)
            predicted_prices = current_close.values * np.exp(y_pred_lr)

            # ── Evaluate metrics (log-return + price MAPE) ────
            metrics_df      = _evaluate_metrics(y_test_lr, y_pred_lr,
                                                actual_prices, predicted_prices,
                                                tf_days)
            current_metrics = metrics_df.iloc[0]

            # ── Save log-return predictions ───────────────────
            pd.DataFrame({
                'Actual_Log_Return':    y_test_lr,
                'Predicted_Log_Return': y_pred_lr,
            }, index=test_dates).to_csv(predictions_path)
            print(f"  Predictions saved  →  {predictions_path}")

            all_metrics = pd.concat([all_metrics, metrics_df], ignore_index=True)
            all_metrics.drop_duplicates(subset=['Timeframe'], keep='last', inplace=True)
            all_metrics.to_csv(metrics_summary_path, index=False)
            print(f"  Metrics summary updated  →  {metrics_summary_path}")

        # ── 4c. PLOT CHARTS (always — after train or resume) ──
        if (actual_prices is not None and predicted_prices is not None
                and test_dates is not None and current_metrics is not None):
            _save_forecast_chart(
                test_dates, actual_prices, predicted_prices,
                current_metrics, cfg.coin, tf_days, chart_path,
            )
            _save_logreturn_chart(
                test_dates, y_test_lr, y_pred_lr,
                current_metrics, cfg.coin, tf_days, lr_chart_path,
            )



# ==============================================================================
# MAIN
# ==============================================================================

def main():
    drive.mount('/content/drive', force_remount=True)

    cfg = Config()
    for d in [cfg.raw_dir, cfg.proc_dir, cfg.feat_sel_dir, cfg.lstm_out_dir]:
        os.makedirs(d, exist_ok=True)

    df_raw       = crawl_raw_data(cfg)
    df_engineered = preprocess_and_engineer(df_raw, cfg)
    run_feature_selection(df_engineered, cfg)
    train_and_evaluate_lstm(cfg)

    print("\nPROCESS COMPLETED SUCCESSFULLY.")


if __name__ == '__main__':
    main()

Mounted at /content/drive
--- STEP 1: CRAWLING RAW DATA ---
Raw data already exists. Loading from disk.

--- STEP 2: PRE-PROCESSING & ENGINEERING ---
Engineered data already exists. Loading from disk.

--- STEP 3: FEATURE SELECTION ---

  FEATURE SELECTION — TF1d
  Golden features already exist for TF1d — skipping.
  (Delete the file to re-run this timeframe.)

  FEATURE SELECTION — TF7d
  Golden features already exist for TF7d — skipping.
  (Delete the file to re-run this timeframe.)

  FEATURE SELECTION — TF14d
  Golden features already exist for TF14d — skipping.
  (Delete the file to re-run this timeframe.)

  FEATURE SELECTION COMPLETE

--- STEP 4: LSTM MODELING ---

--- TIMEFRAME: 1 day(s) ---
  Found saved predictions — loading from disk …
  Metrics already on record — skipping re-evaluation.
  Forecast chart saved  →  /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/LSTM_Results/bitcoin_LSTM_forecast_chart_TF1.png
  Log-return chart saved  →  /content/drive/MyDr

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  Training LSTM …  (max 100 epochs, patience=20)
  Stopped at epoch 41 / 100
  Loss chart saved  →  /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/LSTM_Results/bitcoin_LSTM_loss_chart_TF7.png

───────────────────────────────────────────────
  REGRESSION METRICS  |  horizon = 7d
───────────────────────────────────────────────
  [Log-Return Space]
  MAE   :     0.082917
  RMSE  :     0.105458
  R²    :      -1.9016
  [Price Space]
  MAPE  :        8.12%
───────────────────────────────────────────────

  Predictions saved  →  /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/LSTM_Results/bitcoin_LSTM_Predictions_TF7.csv
  Metrics summary updated  →  /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/LSTM_Results/bitcoin_LSTM_Evaluation_Summary.csv
  Forecast chart saved  →  /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/LSTM_Results/bitcoin_LSTM_forecast_chart_TF7.png
  Log-return chart saved  →  /content/drive/MyDrive/Crypt

In [ ]:
# Calculate top 10 features by importance for TF7 and generate descriptives

import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor

# Define file paths and variables
base_dir = '/content/drive/MyDrive/Crypto Research/DATA/Technical Indicators'
feat_dir = os.path.join(base_dir, 'Features_Selection_Golden')
out_dir = os.path.join(base_dir, 'Descriptives')
input_file = os.path.join(feat_dir, 'bitcoin_GoldenFeatures_TF7.csv')
output_file = os.path.join(out_dir, 'bitcoin_descriptives_TF7_top10.csv')
train_end = '2023-12-31'
tf = 7

# Ensure output directory exists
os.makedirs(out_dir, exist_ok=True)

# Load the dataset
df = pd.read_csv(input_file, index_col='Date', parse_dates=True)

# Drop target and raw price columns from the features list
drop_cols = {'y_raw_close_price', f'Target_{tf}d', 'Target_1d', 'Target_14d'}
feat_cols = [c for c in df.columns if c not in drop_cols]

# Prepare training data
X = df[feat_cols].replace([np.inf, -np.inf], np.nan).dropna()
y = df.loc[X.index, f'Target_{tf}d']
train_mask = X.index <= train_end
X_tr, y_tr = X[train_mask], y[train_mask]

# Fit Random Forest to find top 10 important features
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)
top10_features = pd.Series(rf.feature_importances_, index=feat_cols).nlargest(10).index.tolist()

# Generate descriptives for only the top 10 important features
descriptives_table = df[top10_features].describe().T.round(4)

# Save the table to a CSV file
descriptives_table.to_csv(output_file)

# Display completion message and table
print(f'Saved top 10 importance descriptives table to: {output_file}')
print(descriptives_table)

Saved top 10 importance descriptives table to: /content/drive/MyDrive/Crypto Research/DATA/Technical Indicators/Descriptives/bitcoin_descriptives_TF7_top10.csv
                               count    mean       std         min     25%  \
hashrate_14_EMA               4720.0 -0.3726    0.9036     -1.0858 -0.9980   
sentinusd_7_ROC               4720.0 -0.0040    1.5546    -12.2699 -0.1897   
mediantransactionvalue_7_ROC  4720.0 -1.2250  125.2720  -7719.1978 -0.2907   
size_14_SMA                   4720.0 -0.0670    0.7148     -1.7985 -0.1858   
mediantransactionvalue_7_STD  4720.0  0.0022    0.0069      0.0000  0.0004   
activeaddresses_14_ROC        4720.0  3.2461  462.6645 -11546.0461 -9.0891   
transactionfees_14_ROC        4720.0  0.5365  132.6305  -4721.6908 -0.2769   
difficulty_14_STD             4720.0  0.0065    0.0205      0.0000  0.0001   
sentinusd_14_STD              4720.0  0.0029    0.0060      0.0000  0.0002   
transactionfees_7_STD         4720.0  0.0034    0.0081      